# S2 — AI Engineering: Ejercicios de Código

**Módulo:** Semester 2 — AI Engineering  
**Lecciones:** 2.1 Cuándo usar LLM · 2.2 Structured Outputs · 2.5 Caching · 2.6 Evaluación

## Objetivo

Construir los cuatro componentes clave de un pipeline de AI Engineering real:
un árbol de decisión LLM/código, un validador de outputs estructurados,
una caché semántica, y un evaluador LLM-as-judge.

## Instrucciones

- Implementa cada función donde dice `pass` o `raise NotImplementedError`.
- Ejecuta las celdas de tests. Si todos los `assert` pasan sin excepción, el ejercicio está correcto.
- Los tests están diseñados para ejecutarse **sin API key real** usando mocks o lógica local.
- Lee el docstring de cada función antes de implementarla — contiene las restricciones.


In [ ]:
# Dependencias
# %pip install numpy pydantic --quiet

import json
import math
import os
import re
from typing import Any, Optional
from unittest.mock import MagicMock, patch

import numpy as np

# API key (no se necesita para los tests de este notebook)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'sk-test')

print('Entorno listo.')


---
## Ejercicio 1 — Árbol de decisión: LLM vs. código

**Contexto:** Lección 2.1 — Cuándo usar LLM.

La decisión de usar un LLM donde basta con código es uno de los errores más
caros en producción: entre 10x y 100x más caro por operación, más lento y más
difícil de debuggear. El árbol de decisión de la lección tiene tres preguntas.

**Implementa** `classify_task(description: str) -> str` que devuelve:
- `"code"` si la tarea es determinista, auditable o de alto volumen
- `"llm"` si requiere comprensión de lenguaje natural, razonamiento o generación
- `"hybrid"` si tiene ambas características

**Criterios de éxito:** Al menos 6 casos de prueba cubren los tres valores posibles.


In [ ]:
# Señales por categoría
_CODE_SIGNALS = [
    "reglas fijas", "formato estándar", "formato estandar",
    "categorías fijas", "categorias fijas", "validar email",
    "calcular", "ordenar", "consultar por id", "exactitud 100",
    "reproducible", "auditable", "regex", "parsear fecha",
    "enrutar", "precio", "clasificar en categorias fijas",
    "clasificar en categorías fijas",
]
_LLM_SIGNALS = [
    "razonamiento", "interpretar", "generar", "resumir",
    "redactar", "comprender", "explicar", "intención",
    "intencion", "lenguaje natural", "clasificar intención",
    "extraer entidades", "sentimiento", "traducir",
]


def classify_task(description: str) -> str:
    """
    Clasifica si una tarea debe implementarse con código determinista,
    un LLM, o un enfoque híbrido.

    Parámetros
    ----------
    description : str
        Descripción en texto libre de la tarea a realizar.

    Retorna
    -------
    str
        "code"   — tarea determinista, usa código
        "llm"    — requiere comprensión/generación de lenguaje natural
        "hybrid" — parte determinista + parte de razonamiento

    Lógica esperada
    ---------------
    1. Comprueba señales de código (_CODE_SIGNALS) en la descripción.
    2. Comprueba señales de LLM (_LLM_SIGNALS) en la descripción.
    3. Si solo hay señales de código → "code".
    4. Si solo hay señales de LLM → "llm".
    5. Si hay señales de ambos → "hybrid".
    6. Sin señales claras → "hybrid" (decisión conservadora).
    """
    # TODO: implementa la lógica de clasificación usando _CODE_SIGNALS y _LLM_SIGNALS
    raise NotImplementedError


In [ ]:
# --- Tests Ejercicio 1 ---

# Casos CODE
assert classify_task("Validar que el email tiene formato estándar") == "code", \
    "Validación de email es determinista → code"
assert classify_task("Calcular el precio final aplicando descuentos en categorías fijas") == "code", \
    "Cálculo de precios es determinista → code"
assert classify_task("Consultar por id el registro en la base de datos y devolver el precio") == "code", \
    "Consulta por ID es determinista → code"

# Casos LLM
assert classify_task("Resumir el documento de 20 páginas en 3 párrafos") == "llm", \
    "Resumir requiere comprensión → llm"
assert classify_task("Interpretar la intención del mensaje del usuario y generar una respuesta") == "llm", \
    "Interpretar intención → llm"
assert classify_task("Traducir el texto al inglés manteniendo el tono") == "llm", \
    "Traducción con contexto → llm"

# Casos HYBRID
assert classify_task(
    "Interpretar la intención del usuario y luego calcular el precio exacto"
) == "hybrid", "Interpretación + cálculo exacto → hybrid"
assert classify_task(
    "Extraer entidades del texto y luego consultar por id si existen en la BD"
) == "hybrid", "Extracción + consulta determinista → hybrid"

# El resultado debe ser uno de los tres valores válidos
for desc in ["foo bar", "", "algo sin señales claras"]:
    result = classify_task(desc)
    assert result in {"code", "llm", "hybrid"}, f"Valor inesperado '{result}' para '{desc}'"

print("✅ Ejercicio 1: todos los tests pasan.")


---
## Ejercicio 2 — Structured output validator

**Contexto:** Lección 2.2 — Structured Outputs y Function Calling.

El LLM devuelve texto. El resto del sistema necesita JSON tipado. Sin una capa
de validación explícita, los errores de formato llegan silenciosamente a producción.
El antipatrón clásico: confiar en que el LLM "siempre" devuelve JSON válido.

**Implementa** `validate_llm_output(raw_output: str)` que parsea y valida
el output del LLM contra el schema de `InvoiceExtraction`.

**Schema esperado:**
```
{
  "company_name": str,
  "date": str,          # formato YYYY-MM-DD
  "total_amount": float,
  "line_items": [
    {"description": str, "amount": float}
  ]
}
```

**Criterios de éxito:** Input válido, campo faltante, tipo incorrecto, JSON no válido.


In [ ]:
def validate_llm_output(raw_output: str) -> tuple[bool, Optional[dict], Optional[str]]:
    """
    Parsea y valida el output de texto del LLM contra el schema InvoiceExtraction.

    Parámetros
    ----------
    raw_output : str
        Texto devuelto por el LLM. Puede ser JSON puro, JSON dentro de
        un bloque de código markdown (```json ... ```), o texto inválido.

    Retorna
    -------
    (is_valid, parsed_data, error)
        is_valid   : bool   — True si el output cumple el schema
        parsed_data: dict | None — datos parseados si is_valid es True
        error      : str  | None — mensaje de error si is_valid es False

    Validaciones requeridas
    -----------------------
    1. El texto es JSON parseable (extrae JSON de bloques markdown si es necesario).
    2. Campos obligatorios presentes: company_name, date, total_amount, line_items.
    3. company_name y date son str no vacíos.
    4. total_amount es int o float (convierte int a float).
    5. line_items es una lista, cada item tiene 'description' (str) y 'amount' (float/int).
    """
    # TODO: implementa la extracción y validación
    # Pista: usa re.search(r'```json\s*([\s\S]*?)```', raw_output) para bloques markdown
    # Pista: json.loads() lanza json.JSONDecodeError si el texto no es JSON válido
    raise NotImplementedError


In [ ]:
# --- Tests Ejercicio 2 ---

VALID_JSON = json.dumps({
    "company_name": "Acme Corp",
    "date": "2024-03-15",
    "total_amount": 1250.0,
    "line_items": [
        {"description": "Consultoría", "amount": 1000.0},
        {"description": "Materiales", "amount": 250.0},
    ]
})

# Caso 1: input completamente válido
ok, data, err = validate_llm_output(VALID_JSON)
assert ok is True, "JSON válido debe pasar validación"
assert data is not None
assert data["company_name"] == "Acme Corp"
assert isinstance(data["total_amount"], float)
assert err is None

# Caso 2: JSON dentro de bloque markdown
markdown_output = f"```json\n{VALID_JSON}\n```"
ok2, data2, err2 = validate_llm_output(markdown_output)
assert ok2 is True, "Debe extraer JSON de bloque markdown"
assert data2["company_name"] == "Acme Corp"

# Caso 3: campo obligatorio faltante
missing_field = json.dumps({
    "company_name": "Acme Corp",
    "date": "2024-03-15",
    # total_amount ausente
    "line_items": []
})
ok3, data3, err3 = validate_llm_output(missing_field)
assert ok3 is False, "Campo faltante debe fallar"
assert data3 is None
assert err3 is not None and len(err3) > 0

# Caso 4: tipo incorrecto en total_amount
wrong_type = json.dumps({
    "company_name": "Acme Corp",
    "date": "2024-03-15",
    "total_amount": "mil doscientos",  # string, no numérico
    "line_items": []
})
ok4, data4, err4 = validate_llm_output(wrong_type)
assert ok4 is False, "total_amount string debe fallar"

# Caso 5: no es JSON en absoluto
ok5, data5, err5 = validate_llm_output("Aquí está la información de la factura: Acme Corp...")
assert ok5 is False, "Texto plano debe fallar"
assert err5 is not None

# Caso 6: total_amount como int (debe convertirse a float)
int_amount = json.dumps({
    "company_name": "Beta SA",
    "date": "2024-01-01",
    "total_amount": 500,  # int, válido
    "line_items": [{"description": "Servicio", "amount": 500}]
})
ok6, data6, err6 = validate_llm_output(int_amount)
assert ok6 is True, "total_amount int debe ser válido (convertir a float)"
assert isinstance(data6["total_amount"], float)

print("✅ Ejercicio 2: todos los tests pasan.")


---
## Ejercicio 3 — Semantic cache

**Contexto:** Lección 2.5 — Caching y optimización.

Una caché semántica reutiliza respuestas cuando una nueva query es
semánticamente equivalente a una anterior, aunque las palabras sean distintas.
El umbral de similitud es el parámetro clave: demasiado bajo → falsos positivos;
demasiado alto → miss rate elevado. En producción, 0.92–0.95 es habitual para
queries de soporte o FAQ.

**Implementa** `SemanticCache` con:
- `embed(text)` → vector numpy normalizado (bag-of-words para tests sin API)
- `add(query, response)` → añade al cache
- `get(query)` → respuesta si similitud > threshold, o `None`

**Criterios de éxito:** Hit en query idéntica, hit en query similar, miss en query distinta.


In [ ]:
class SemanticCache:
    """
    Caché semántica que reutiliza respuestas cuando la similitud coseno
    entre la query nueva y alguna query almacenada supera el umbral.

    Parámetros
    ----------
    threshold : float
        Similitud coseno mínima para considerar un hit (0.0–1.0).
        Por defecto 0.92.
    embed_fn : callable, opcional
        Función de embedding externa. Si es None, usa embed() interno
        (bag-of-words normalizado), útil para tests sin API.
    """

    def __init__(self, threshold: float = 0.92, embed_fn=None):
        self.threshold = threshold
        self._embed_fn = embed_fn
        # TODO: inicializa la estructura de almacenamiento
        # Sugerencia: lista de (vector, response) o dict de str→(vector, response)
        raise NotImplementedError

    def embed(self, text: str) -> np.ndarray:
        """
        Embedding local basado en bag-of-words normalizado.
        Sirve para tests sin necesidad de API.

        Algoritmo:
        1. Tokeniza el texto en palabras (lowercase, split).
        2. Construye un vocabulario global (palabras únicas vistas hasta ahora).
        3. Devuelve un vector de conteos normalizado a longitud 1 (norma L2).

        Si se proporcionó embed_fn en el constructor, úsala en su lugar.
        """
        # TODO: implementa el embedding bag-of-words con vocabulario dinámico
        # Pista: mantén self._vocab como un dict {word: index}
        # Pista: np.linalg.norm para normalizar
        raise NotImplementedError

    @staticmethod
    def _cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        """
        Similitud coseno entre dos vectores numpy.
        Si alguno es el vector cero, devuelve 0.0.
        """
        # TODO: implementa similitud coseno
        # Fórmula: dot(a, b) / (norm(a) * norm(b))
        # Pista: comprueba norm > 0 para evitar división por cero
        raise NotImplementedError

    def add(self, query: str, response: str) -> None:
        """
        Añade un par (query, response) al cache.
        Calcula y almacena el embedding de query para comparaciones futuras.
        """
        # TODO: implementa add
        raise NotImplementedError

    def get(self, query: str) -> Optional[str]:
        """
        Busca en el cache la respuesta más similar a query.

        Retorna
        -------
        str | None
            La respuesta del entry más similar si su similitud > threshold.
            None si ninguna entrada supera el umbral.
        """
        # TODO: implementa get
        # Pista: itera sobre las entradas, calcula similitud, guarda el máximo
        raise NotImplementedError

    def __len__(self) -> int:
        """Número de entradas en el cache."""
        # TODO
        raise NotImplementedError


In [ ]:
# --- Tests Ejercicio 3 ---

cache = SemanticCache(threshold=0.85)

# Cache empieza vacío
assert len(cache) == 0
assert cache.get("hola") is None, "Cache vacío debe devolver None"

# Añadir una entrada
cache.add("cuál es el precio del plan básico", "El plan básico cuesta 9,99€/mes.")
assert len(cache) == 1

# Hit exacto: misma query
result = cache.get("cuál es el precio del plan básico")
assert result == "El plan básico cuesta 9,99€/mes.", f"Hit exacto esperado, got: {result}"

# Hit semántico: query ligeramente diferente pero vocabulario muy similar
result_similar = cache.get("precio del plan básico cuál es")
assert result_similar == "El plan básico cuesta 9,99€/mes.", \
    "Query con mismas palabras reordenadas debe ser hit"

# Miss: query completamente distinta
result_miss = cache.get("cómo cancelo mi suscripción")
assert result_miss is None, f"Query distinta debe ser miss, got: {result_miss}"

# Threshold alto: miss forzado con umbral 0.999 (solo coincidencia exacta)
strict_cache = SemanticCache(threshold=0.999)
strict_cache.add("cuál es el precio del plan básico", "El plan básico cuesta 9,99€/mes.")
result_strict = strict_cache.get("precio del plan básico")
# Con umbral muy alto, una query con pocas palabras distintas puede ser miss
# (comportamiento aceptable: None o la respuesta si similitud == 1.0 exacto)
assert result_strict in (None, "El plan básico cuesta 9,99€/mes."), \
    "Con threshold muy alto, resultado debe ser None o exact match"

# Múltiples entradas: devuelve la más similar
cache2 = SemanticCache(threshold=0.5)
cache2.add("precio plan básico", "Básico: 9,99€")
cache2.add("precio plan premium", "Premium: 29,99€")
r = cache2.get("cuánto cuesta el plan premium")
assert r == "Premium: 29,99€", f"Debe recuperar el más similar, got: {r}"

# cosine_similarity de vectores iguales = 1.0
v = np.array([1.0, 0.0, 0.0])
assert abs(SemanticCache._cosine_similarity(v, v) - 1.0) < 1e-9

# cosine_similarity de vectores ortogonales = 0.0
v2 = np.array([0.0, 1.0, 0.0])
assert abs(SemanticCache._cosine_similarity(v, v2) - 0.0) < 1e-9

# cosine_similarity con vector cero = 0.0
v_zero = np.array([0.0, 0.0, 0.0])
assert SemanticCache._cosine_similarity(v, v_zero) == 0.0

print("✅ Ejercicio 3: todos los tests pasan.")


---
## Ejercicio 4 — LLM-as-judge evaluator

**Contexto:** Lección 2.6 — Evaluación básica.

El LLM-as-judge es el método que escala mejor cuando las heurísticas
programáticas no son suficientes. El riesgo principal: asumir que el juez
es correcto sin calibrarlo contra criterio humano. Este ejercicio construye
el evaluador con rúbrica ponderada y estadísticas de batch.

**Implementa:**
- `build_eval_prompt(question, response, rubric)` → prompt de evaluación
- `parse_scores(llm_output, rubric)` → dict de scores por criterio
- `evaluate_response(question, response, rubric, llm_fn)` → weighted score [0, 1]
- `batch_evaluate(qa_pairs, rubric, llm_fn)` → estadísticas del batch

**Criterios de éxito:** Tests con mock del LLM. Weighted score correcto dado scores conocidos.


In [ ]:
# Rúbrica: {criterio: peso}. Los pesos deben sumar 1.0.
DEFAULT_RUBRIC = {
    "relevance": 0.4,    # ¿La respuesta responde la pregunta?
    "accuracy": 0.3,     # ¿La información es correcta?
    "conciseness": 0.3,  # ¿Es concisa sin perder información clave?
}


def build_eval_prompt(question: str, response: str, rubric: dict) -> str:
    """
    Construye el prompt que se enviará al LLM-juez.

    El prompt debe:
    1. Presentar la pregunta original y la respuesta a evaluar.
    2. Listar cada criterio del rubric con su descripción.
    3. Pedir al LLM que puntúe del 1 al 5 cada criterio.
    4. Especificar el formato de salida esperado: JSON con
       {criterio: {"score": int, "justification": str}} para cada criterio.

    Retorna
    -------
    str : el prompt completo listo para enviar al LLM.
    """
    # TODO: construye el prompt
    # Asegúrate de que el formato de salida pedido sea JSON parseable
    raise NotImplementedError


def parse_scores(llm_output: str, rubric: dict) -> dict[str, float]:
    """
    Parsea el output del LLM-juez y extrae los scores por criterio.

    El LLM puede devolver:
    - JSON puro: {"relevance": {"score": 4, ...}, ...}
    - JSON dentro de un bloque markdown ```json ... ```

    Retorna
    -------
    dict[str, float]
        {criterio: score_normalizado} donde score_normalizado = score/5.0.
        Si el criterio no aparece en el output, asigna 0.0.
        Si el JSON no es parseable, devuelve {criterio: 0.0} para todos.
    """
    # TODO: implementa parse_scores
    # Pista: usa re.search para extraer JSON de bloques markdown
    # Pista: los scores del LLM son 1-5; normaliza dividiendo por 5.0
    raise NotImplementedError


def evaluate_response(
    question: str,
    response: str,
    rubric: dict,
    llm_fn,  # callable(prompt: str) -> str
) -> float:
    """
    Evalúa una respuesta usando el LLM-juez y la rúbrica ponderada.

    Parámetros
    ----------
    llm_fn : callable
        Función que acepta un str (prompt) y devuelve un str (respuesta del LLM).
        En producción sería la llamada real a la API; en tests, un mock.

    Retorna
    -------
    float : weighted_score en [0.0, 1.0]
        Suma de (score_normalizado_criterio * peso_criterio) para cada criterio.
    """
    # TODO: implementa evaluate_response usando build_eval_prompt, llm_fn y parse_scores
    raise NotImplementedError


def batch_evaluate(
    qa_pairs: list[dict],  # [{"question": str, "response": str}, ...]
    rubric: dict,
    llm_fn,
) -> dict:
    """
    Evalúa un batch de pares pregunta-respuesta.

    Retorna
    -------
    dict con:
        "scores"    : list[float] — score de cada par
        "mean"      : float       — media de los scores
        "min"       : float       — score mínimo
        "max"       : float       — score máximo
        "pct_above_threshold" : float — porcentaje con score > 0.7 (0.0–1.0)
    """
    # TODO: implementa batch_evaluate
    raise NotImplementedError


In [ ]:
# --- Tests Ejercicio 4 ---

# Mock del LLM que devuelve scores conocidos
def make_mock_llm(scores_by_criterion: dict) -> callable:
    """
    Crea un mock de LLM que devuelve siempre los scores especificados.
    scores_by_criterion: {criterio: score_1_to_5}
    """
    output = json.dumps({
        k: {"score": v, "justification": "test"}
        for k, v in scores_by_criterion.items()
    })
    return lambda prompt: output


# Test build_eval_prompt: debe contener la pregunta, la respuesta y los criterios
prompt = build_eval_prompt(
    question="¿Cuál es el precio del plan básico?",
    response="El plan básico cuesta 9,99€ al mes.",
    rubric=DEFAULT_RUBRIC,
)
assert isinstance(prompt, str) and len(prompt) > 50, "Prompt debe ser un string no vacío"
assert "relevance" in prompt.lower() or "relevancia" in prompt.lower(), \
    "Prompt debe mencionar el criterio 'relevance'"
assert "¿Cuál es el precio" in prompt, "Prompt debe incluir la pregunta"
assert "9,99€" in prompt, "Prompt debe incluir la respuesta a evaluar"

# Test parse_scores: scores 4/5 → 0.8
mock_output = json.dumps({
    "relevance": {"score": 4, "justification": "buena relevancia"},
    "accuracy": {"score": 5, "justification": "muy precisa"},
    "conciseness": {"score": 3, "justification": "algo larga"},
})
scores = parse_scores(mock_output, DEFAULT_RUBRIC)
assert abs(scores["relevance"] - 0.8) < 1e-6, f"4/5 = 0.8, got {scores['relevance']}"
assert abs(scores["accuracy"] - 1.0) < 1e-6, f"5/5 = 1.0, got {scores['accuracy']}"
assert abs(scores["conciseness"] - 0.6) < 1e-6, f"3/5 = 0.6, got {scores['conciseness']}"

# Test parse_scores: JSON en bloque markdown
md_output = f"```json\n{mock_output}\n```"
scores_md = parse_scores(md_output, DEFAULT_RUBRIC)
assert abs(scores_md["relevance"] - 0.8) < 1e-6, "Debe parsear JSON en bloque markdown"

# Test parse_scores: JSON inválido → todos 0.0
bad_scores = parse_scores("esto no es json", DEFAULT_RUBRIC)
assert all(v == 0.0 for v in bad_scores.values()), "JSON inválido → scores 0.0"

# Test evaluate_response: weighted score correcto
# relevance=4/5*0.4 + accuracy=5/5*0.3 + conciseness=3/5*0.3 = 0.32 + 0.30 + 0.18 = 0.80
mock_llm = make_mock_llm({"relevance": 4, "accuracy": 5, "conciseness": 3})
score = evaluate_response(
    question="¿Cuál es el precio?",
    response="El precio es 9,99€.",
    rubric=DEFAULT_RUBRIC,
    llm_fn=mock_llm,
)
expected = 4/5 * 0.4 + 5/5 * 0.3 + 3/5 * 0.3
assert abs(score - expected) < 1e-6, f"Weighted score esperado {expected:.4f}, got {score:.4f}"
assert 0.0 <= score <= 1.0, "Score debe estar en [0, 1]"

# Test batch_evaluate
high_llm = make_mock_llm({"relevance": 5, "accuracy": 5, "conciseness": 5})  # score = 1.0
low_llm_scores = {"relevance": 2, "accuracy": 2, "conciseness": 2}  # score = 0.4

# Alternamos high/low usando contador en closure
call_count = {"n": 0}
high_out = json.dumps({k: {"score": 5, "justification": ""} for k in DEFAULT_RUBRIC})
low_out  = json.dumps({k: {"score": 2, "justification": ""} for k in DEFAULT_RUBRIC})
def alternating_llm(prompt):
    call_count["n"] += 1
    return high_out if call_count["n"] % 2 == 1 else low_out

qa_pairs = [
    {"question": "Q1", "response": "R1"},
    {"question": "Q2", "response": "R2"},
    {"question": "Q3", "response": "R3"},
    {"question": "Q4", "response": "R4"},
]
stats = batch_evaluate(qa_pairs, DEFAULT_RUBRIC, alternating_llm)

assert "scores" in stats and len(stats["scores"]) == 4
assert "mean" in stats
assert "min" in stats and "max" in stats
assert "pct_above_threshold" in stats
assert abs(stats["max"] - 1.0) < 1e-6, f"Max score debe ser 1.0, got {stats['max']}"
assert abs(stats["min"] - 0.4) < 1e-6, f"Min score debe ser 0.4, got {stats['min']}"
# 2 de 4 pares tienen score 1.0 > 0.7 → pct = 0.5
assert abs(stats["pct_above_threshold"] - 0.5) < 1e-6, \
    f"50% por encima de 0.7, got {stats['pct_above_threshold']}"

print("✅ Ejercicio 4: todos los tests pasan.")
print(f"   Weighted score ejemplo: {score:.3f} (esperado {expected:.3f})")
print(f"   Batch stats: mean={stats['mean']:.2f}, min={stats['min']:.2f}, max={stats['max']:.2f}, pct_above_0.7={stats['pct_above_threshold']:.0%}")


---
## Referencias y tips

### Lecciones relacionadas

| Ejercicio | Lección |
|---|---|
| 1 — Árbol decisión | **2.1** Cuándo usar LLM: el árbol de tres preguntas |
| 2 — Structured output validator | **2.2** Structured outputs: la capa de validación como contrato |
| 3 — Semantic cache | **2.5** Caching: estático arriba, dinámico abajo, semántico para FAQs |
| 4 — LLM-as-judge | **2.6** Evaluación: los 3 métodos y cuándo usar cada uno |

### Tips de implementación

**Ejercicio 1:** La clasificación basada en señales de palabras clave es un buen punto de partida, pero en producción querrás añadir un nivel de LLM para los casos ambiguos. El árbol de decisión cubre el 80% de los casos con costo cero.

**Ejercicio 2:** `json.JSONDecodeError` es tu mejor amigo para detectar outputs inválidos. Para producción real, considera `pydantic` con `model_validate()` en lugar de validación manual — es más robusto y genera mensajes de error legibles.

**Ejercicio 3:** El embedding bag-of-words es suficiente para tests pero no para producción. En producción usa `text-embedding-3-small` de OpenAI o `sentence-transformers`. El threshold óptimo depende de tu distribución de queries — mídelo con un golden set.

**Ejercicio 4:** El mayor riesgo del LLM-as-judge es el sesgo de posición y el sesgo de longitud (el juez tiende a puntuar más alto las respuestas largas). Calibra siempre contra 50-100 evaluaciones humanas antes de confiar en el juez como métrica de release.

### Próximos pasos

- **S3 Agent Engineering:** los verifiers del Ejercicio 2 son la base de los `verifier nodes` en agentes.
- **S5 Context Engineering:** el semantic cache del Ejercicio 3 es RAG simplificado — en S5 construyes la versión completa con chunking y reranking.
